# POIの影響範囲をPythonで推定する
固定半径・適応半径を比較する部分追試です。吉祥寺4km四方で、POIから推定した機能的影響を見ます。実際の街・商圏の正しい境界を求めるものではありません。

[方法と論文との差分](../docs/urban-function.md) / [検証](../docs/urban-function-validation.md) / [記事引き継ぎ](../docs/urban-function-article-handoff.md)

## 1. 固定入力と住宅の監査
Overture 2026-09-23.0、N02-25駅代表点、EPSG:6677を再利用します。住宅POIは1件だけです。住宅なしとは解釈せず、住宅の帰属度は未推定とします。新規ネットワーク取得は実行しません。

In [ ]:
from geoai_open_lab.urban_function import run, OUT
from geoai_open_lab.urban_function_figures import render
from IPython.display import SVG, display
import pandas as pd
report, comparison, external = run()
pd.Series(report["function_counts"], name="POI登録数")

## 2. 3方式を同じ入力で比較
区画内集計・全POI共通のガウス半径・機能カテゴリごとのKNNから作る適応半径を比較します。S=1、C=0.5S+0.5Dは今回の独自ベースラインです。S1の重みは未確認。機能別min–max正規化後に帰属度を計算するため、単純な件数比ではありません。

In [ ]:
render()
display(SVG(filename=str(OUT / "input_audit.svg")))
display(SVG(filename=str(OUT / "membership.svg")))

## 3. 強度・混合・横断線
住宅を無理に補わず、2機能の混色を表示します。混合度はエントロピーの連続値であり、補足S2の未確認の閾値を使う遷移帯分類は実装していません。

In [ ]:
for name in ("intensity", "dominant_mixing", "transect", "difference"):
    display(SVG(filename=str(OUT / (name + ".svg"))))

## 4. 安定性と外部対応は別の問い
k=3,5,7,10、半径0.6～1.4倍、100m/250mを比較します。MAEは帰属度0～1の単位。解像度間は共通50m区画へ面積対応させます。土地利用2021の公園・緑地との対応は限定的な代理検証で、住宅・商業の精度検証ではありません。

In [ ]:
display(comparison)
display(external)

## 5. 分かったことと言えないこと
住宅入力が不足する状態では商業・住宅・緑地の遷移を推定できません。適応半径の緑地等への偏り、入力の切取りによる端効果、機能別正規化の影響を残します。独立土地利用との一致は低く、精度改善を確認したとは言えません。人気・利用者数・需要・売上へ読み替えないでください。Sや重み、カテゴリを変える研究は別条件として扱います。